# Set up for State of Health Estimation
Create the estimator, which requires complete with the initial values for parameters and their uncertainities.

In [1]:
%matplotlib inline
from matplotlib import pyplot as plt
from moirae.estimators.online.filters.distributions import MultivariateGaussian
from moirae.estimators.online.joint import JointEstimator
from moirae.models.ecm import EquivalentCircuitModel
from moirae.models.ecm.advancedSOH import ECMASOH
from moirae.models.ecm.transient import ECMTransientVector
from moirae.models.ecm.ins_outs import ECMInput, ECMMeasurement
from moirae.interface import run_online_estimate
from batdata.data import BatteryDataset
from pathlib import Path
from tqdm import tqdm
import pickle as pkl
import numpy as np

Configuration

In [2]:
estimate_dir = Path('estimates')
estimate_dir.mkdir(exist_ok=True)

In [3]:
initial_asoh = ECMASOH.model_validate_json(Path('initial-asoh.json').read_text())
initial_asoh.ocv(0.5) # quick and dirty init for soc_pinpoints
# fix reference OCV
soc_pinpoints = [-0.1] + initial_asoh.ocv.ocv_ref.soc_pinpoints.flatten().tolist() + [1.1]
base_vals = [0.] + initial_asoh.ocv.ocv_ref.base_values.flatten().tolist() + [6.5]
initial_asoh.ocv.ocv_ref.base_values = np.array([base_vals])
initial_asoh.ocv.ocv_ref.soc_pinpoints = np.array(soc_pinpoints)
initial_asoh.ocv.ocv_ref.interpolation_style='linear'
initial_asoh.mark_updatable('r0.base_values')
initial_asoh.mark_updatable('q_t.base_values')
print(f'Preparing to update {initial_asoh.num_updatable} parameters from: {initial_asoh.updatable_names}')

Preparing to update 10 parameters from: ('q_t.base_values', 'r0.base_values')


## Find cells
Find all the runs then pull out one at example

In [4]:
all_cells = sorted(Path('synth-data').glob('*.hdf5'))
print(f'Found {len(all_cells)}. Will use {all_cells[0].name} as an example')

Found 256. Will use bilinear-0.hdf5 as an example


In [5]:
example_data = BatteryDataset.from_batdata_hdf(all_cells[0])
print(f'Loaded a cell with {len(example_data.raw_data)} current and voltage measurements.')

Loaded a cell with 1362215 current and voltage measurements.


C:\Users\lward\AppData\Local\miniconda3\envs\rovidemo\lib\site-packages\batdata\data.py:90: UserWarning: Metadata was created in a different version of batdata. supplied=0.3.2, current=0.3.3.
  warnings.warn(f'Metadata was created in a different version of batdata. supplied={supplied_version}, current={__version__}.')


## Prepare Estimation Function
We need a function to prepare the [estimator](https://rovi-org.github.io/auto-soh/estimator.html#online-estimators) used for tracking changes in parameters.

In [6]:
def create_estimator(dataset: BatteryDataset):
    """Generate an estimator based on initial parameter estimates

    Args:
        dataset: Dataset on which we are running the estimator. [Not being used for now]
    Returns:
        Estimator ready for use
    """

    # Uncertainties for the parameters
    # For A-SOH, assume 2*standard_dev is 0.5% of the value of the parameter
    asoh_covariance = [(2.5e-03 * initial_asoh.q_t.base_values.item()) ** 2] # +/- std_dev^2 Qt
    asoh_covariance += ((2.5e-03 * initial_asoh.r0.base_values.flatten()) ** 2).tolist() # +/- std_dev^2 of R0
    asoh_covariance = np.diag(asoh_covariance)
    # For the transients, assume SOC is a uniform random variable in [0,1], and hysteresis has 2*std_dev of 1 mV
    init_transients = ECMTransientVector.from_asoh(initial_asoh)
    init_transients.soc = np.atleast_2d(1.)
    tran_covariance = np.diag([1/12, 2.5e-07])

    # Make the noise terms
    #  Logic from: https://github.com/ROVI-org/auto-soh/blob/main/notebooks/demonstrate_joint_ukf.ipynb
    voltage_err = 1.0e-03 # mV voltage error
    noise_sensor = ((voltage_err / 2) ** 2) * np.eye(1)
    noise_asoh = 1.0e-10 * np.eye(asoh_covariance.shape[0])
    noise_tran = 1.0e-08 * np.eye(2)

    return JointEstimator.initialize_unscented_kalman_filter(
        cell_model=EquivalentCircuitModel(),
        initial_asoh=initial_asoh.model_copy(deep=True),
        initial_inputs=ECMInput(
            time=dataset.raw_data['test_time'].iloc[0], 
            current=dataset.raw_data['current'].iloc[0],
        ),
        initial_transients=init_transients,
        covariance_asoh=asoh_covariance,
        covariance_transient=tran_covariance,
        transient_covariance_process_noise=noise_tran,
        asoh_covariance_process_noise=noise_asoh,
        covariance_sensor_noise=noise_sensor
    )

In [7]:
estimator = create_estimator(example_data)

In [8]:
with open('joint-estimator.pkl', 'wb') as fp:
    pkl.dump(estimator, fp)